### Import Dependencies

In [ ]:
import openai
import pandas as pd

from qdrant_client import QdrantClient, models
from qdrant_client.models import (
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    PointStruct,
    Document,
    Prefetch,
    Fusion,
    FusionQuery,
)

import tiktoken

### Create Qdrant collection for hybrid search

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
COLLECTION_NAME = "Amazon-reviews-collection-01"

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE),
        }
    )


In [ ]:
qdrant_client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD,
)

### Embedding Functions

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


In [ ]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
  if len(text_list) <= batch_size:
    response = openai.embeddings.create(
      input=text_list,
      model=model
    )
    return [item.embedding for item in response.data]
  
  all_embeddings = []
  counter = 1
  for i in range(0, len(text_list), batch_size):
    batch = text_list[i:i+batch_size]
    response = openai.embeddings.create(
      input=batch,
      model=model
    )
    all_embeddings.extend([item.embedding for item in response.data])
    print(f"Processed {counter * batch_size} of {len(text_list)}")
    counter += 1

  return all_embeddings

### Read the sampled dataset with Amazon inventory

In [ ]:
from datetime import datetime

from pydantic import BaseModel


class ReviewImage(BaseModel):
    small_image_url: str
    medium_image_url: str
    large_image_url: str
    attachment_type: str


class Review(BaseModel):
    """One Amazon review from the Electronics sample dataset."""

    rating: int
    title: str
    text: str
    images: list[ReviewImage]
    asin: str
    parent_asin: str
    user_id: str
    timestamp: datetime
    helpful_vote: int
    verified_purchase: bool


class EmbeddingPayload(BaseModel):
    """Fields embedded and stored as the Qdrant point payload."""

    preprocessed_data: str
    parent_asin: str

In [ ]:
df_reviews = pd.read_json(
    "../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl",
    lines=True,
)

In [ ]:
df_reviews.head()


In [ ]:
len(df_reviews)

In [ ]:
list(df_reviews["text"].items())[0]

In [ ]:
list(df_reviews["images"].items())[0]

### Preprocess title and features

In [ ]:
def concat_review_title_and_description(review: dict) -> str:
    return f"{review["title"]} {review["text"]}"

In [ ]:
def count_tokens(row: dict) -> int:
    encoding = tiktoken.encoding_for_model("text-embedding-3-small")
    return len(encoding.encode(row["preprocessed_data"]))


In [ ]:
# count_tokens(df_reviews)

In [ ]:
df_reviews["preprocessed_data"] = df_reviews.apply(concat_review_title_and_description, axis=1)
df_reviews["token_count"] = df_reviews.apply(count_tokens, axis=1)


In [ ]:
df_reviews.head()

In [ ]:
len(df_reviews)

In [ ]:
df_reviews = df_reviews[df_reviews["token_count"] < 8192]

In [ ]:
len(df_reviews)

In [ ]:
total_tokens = df_reviews["token_count"].sum()  # pyright: ignore[reportUndefinedVariable]
total_tokens

### Embed the text and additional fields to the payload of each vector for reviews

In [ ]:
df_data_to_embed = df_reviews[list(EmbeddingPayload.__annotations__)]

In [ ]:
df_data_to_embed.head()  # pyright: ignore[reportAttributeAccessIssue]

In [ ]:
reviews_to_embed = [
    EmbeddingPayload.model_validate(record)
    for record in df_data_to_embed.to_dict(orient="records")  # pyright: ignore[reportCallIssue, reportAttributeAccessIssue]
]
reviews_to_embed

In [ ]:
len(reviews_to_embed)

In [ ]:
review_text_to_embed = [payload.preprocessed_data for payload in reviews_to_embed]

In [ ]:
review_text_to_embed

In [ ]:
points_already_ingested = qdrant_client.count(COLLECTION_NAME).count
ingest_needed = points_already_ingested < len(reviews_to_embed)
print(
    f"{points_already_ingested} of {len(reviews_to_embed)} points already in "
    f"'{COLLECTION_NAME}'; ingest_needed = {ingest_needed}"
)

In [ ]:
if ingest_needed:
    embeddings = get_embeddings_batch(review_text_to_embed, batch_size=500)
else:
    embeddings = []
    print("Skipping embedding: collection is already fully ingested.")

In [ ]:
pointstructs: list[PointStruct] = []
for i, (embedding, payload) in enumerate(zip(embeddings, reviews_to_embed), start=1):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding
            },
            payload=payload.model_dump(),
        )
    )

In [ ]:
pointstructs[0].vector if pointstructs else "no pointstructs built (ingest skipped)"

In [ ]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
  batch = pointstructs[i:i+batch_size_qdrant]
  qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=batch,
    wait=True
  )
  counter += 1
  print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")


### A function to run search against reviews on a prefiltered set of product IDs

In [ ]:
from typing import TypedDict
from qdrant_client.models import FieldCondition, Filter, MatchAny


embedding_model = "text-embedding-3-small"

def retrieve_prefiltered_reviews_data(
    query: str, parent_asins: list[str], k=5
):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin", match=MatchAny(any=parent_asins)
                        )
                    ]
                ),
                limit=20,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=k,
    )

    return results



In [ ]:
reviews = retrieve_prefiltered_reviews_data(
    query="last", parent_asins=["B0B9S7TTCZ"]
)
reviews.points

In [ ]:
reviews = retrieve_prefiltered_reviews_data("sound", ["B09V787FQ9", "B0BYMCQQ5Z"])
reviews.points


### Define review retrieval tool

In [ ]:
RetrievedData = TypedDict(
    "RetrievedData",
    {
        "retrieved_asins": list[str],
        "retrieved_reviews": list[str],
        "similarity_scores": list[float],
    },
)


def retrieve_prefiltered_reviews_data(
    query: str, parent_asins: list[str], k=5
) -> RetrievedData:
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin", match=MatchAny(any=parent_asins)
                        )
                    ]
                ),
                limit=20,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=k,
    )

    retrieved_asins = []
    retrieved_reviews = []
    similarity_scores = []

    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint")
        payload = EmbeddingPayload.model_validate(result.payload)
        retrieved_asins.append(payload.parent_asin)
        retrieved_reviews.append(payload.preprocessed_data)
        similarity_scores.append(result.score)

    return {
        "retrieved_asins": retrieved_asins,
        "retrieved_reviews": retrieved_reviews,
        "similarity_scores": similarity_scores,
    }


def process_retrieved_reviews(retrieved_data: RetrievedData) -> str:
    formatted_context = ""

    for asin, review in zip(
        retrieved_data["retrieved_asins"],
        retrieved_data["retrieved_reviews"]
    ):
        formatted_context += f"- ID: {asin}, user review: {review}\n"

    return formatted_context


def get_formatted_reviews_context(query: str, parent_asins: list[str], top_k: int = 5) -> str:
    """Get the top k reviews matching a query for a list of prefiltered items.
    
    Args:
        query: The query to get the top k reviews for
        item_list: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multipple items are prefiltered
    
    Returns:
        A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for a given query.
    """

    retrieved_context = retrieve_prefiltered_reviews_data(query, parent_asins, k=top_k)

    formatted_context = process_retrieved_reviews(retrieved_context)
    return formatted_context


In [ ]:
result = get_formatted_reviews_context("sound", ["B09V787FQ9", "B0BYMCQQ5Z"])
result
print(result)